# LPPLS — Backtest nommé par actif

**Exploratoire** — LPPLS par recherche sur grille. Un signal n’est pas une preuve de bulle. Données propres et cours ajustés recommandés.

## Configuration

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'lppls_core.py').exists() and (ROOT.parent / 'lppls_core.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0,str(ROOT))
from lppls_core import load_prices, to_weekly, write_snapshot_outputs, run_backtest
INPUT_CSV = ROOT / 'Miscellanous.csv'  # modifier pour utiliser un autre CSV
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print('Source:',INPUT_CSV,'| Sortie:',OUTPUT_DIR)

## Importation avec noms de colonnes conservés

In [ ]:
prices = load_prices(INPUT_CSV)
weekly = to_weekly(prices)
print('Dates:', weekly.index.min(), 'à', weekly.index.max())
print('Actifs:', len(weekly.columns),'| semaines:', len(weekly))
weekly.tail()

## LPPLS, classification et portefeuille

In [ ]:
# Grille exhaustive 7×7×7 par actif et fenêtre ; peut prendre plusieurs minutes
outputs = write_snapshot_outputs(weekly, OUTPUT_DIR, grid='full')
print('Fichiers:', sorted(p.name for p in OUTPUT_DIR.iterdir()))
outputs['bubble_type_by_asset.csv'].head()

## Backtest : poids par actif, matrices wide et états

In [ ]:
# Rééquilibrage toutes les 4 semaines, données disponibles au jour de décision uniquement.
# Exécution possible sur la totalité de l'historique après START_DATE.
START_DATE='2024-02-01'
REBALANCE_WEEKS=4
COSTS_BPS=0.0  # un coût ex. 10 bps est possible
metrics=run_backtest(weekly,OUTPUT_DIR,start_date=START_DATE,
                     rebalance_weeks=REBALANCE_WEEKS,costs_bps=COSTS_BPS,grid='tiny')
metrics

## Vérification des sorties

In [ ]:
import pandas as pd
weights=pd.read_csv(OUTPUT_DIR/'backtest_portfolios_long_only_weights.csv')
assert set(weights['asset']).issubset(set(weekly.columns))
print('Actifs du CSV conservés dans la table de poids: OK')
weights.head()